[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *Pandas*.

El proyecto [*Pandas*](https://pandas.pydata.org/) es una herramienta especializada en la gestión de "series" y "*dataframes*", utilizándolos como materia prima para la realización de operaciones de manipulación, transformación y análisis de datos.


*Pandas* cuenta con las siguientes funcionalidades.


* Hace uso intensivo de series y *dataframes*.
* Realiza operaciones de lectura y escritura de datos entre estructuras en memoria y diversos formatos de archivos y bases de datos.
* Alineación de datos y manejo de datos faltantes.
* Modificación de conjuntos de datos (*datasets*).
* Manejo de series de tiempo.

**NOTA** Por convención, el paquete ```pandas``` es importado con el nombre de ```pd```. A lo largo de este curso, se utilizará dicha convención.

## Nota sobre el caso de uso de *Pandas*.

*Pandas* es una biblioteca **madura, estable y ampliamente adoptada** en la industria y la academia. Es la opción predeterminada para la mayoría de las tareas de análisis de datos en *Python* y cuenta con un ecosistema enorme de integraciones.

Sin embargo, tiene limitaciones importantes que conviene conocer:

| Limitación | Descripción |
|---|---|
| **Procesamiento en memoria** | El *dataframe* completo debe caber en la RAM disponible. |
| **Ejecución en un solo hilo** | La mayoría de sus operaciones no aprovechan múltiples núcleos del procesador. |
| **Rendimiento en escala** | A partir de decenas de millones de renglones, el rendimiento se degrada notablemente. |
| **Evaluación ansiosa** | Cada operación se ejecuta de inmediato, sin optimizar el plan de consulta completo. |

Para necesidades modernas donde estas limitaciones importan, existen alternativas más adecuadas:

* [**Polars**](https://pola.rs/): biblioteca de *dataframes* escrita en *Rust*, con evaluación perezosa (*lazy evaluation*), ejecución paralela y rendimiento muy superior en conjuntos de datos grandes. Se cubre en los notebooks **34** y **35**.
* [**PyArrow**](https://arrow.apache.org/docs/python/): implementación en *Python* del formato columnar *Apache Arrow*. Ofrece interoperabilidad eficiente entre herramientas y sirve de base para el backend moderno de *Pandas* (`pd.options.mode.dtype_backend = "pyarrow"`). Se cubre en el notebook **33**.

> *Pandas* sigue siendo la elección correcta para la mayoría de los proyectos cotidianos. Conocer sus límites permite decidir cuándo conviene migrar a una herramienta más especializada.

In [ ]:
%pip install pandas

In [ ]:
import pandas as pd
import numpy as np

## *Pandas* 3 y sus diferencias con *Pandas* 2.

Este curso usa *Pandas* 3, publicado en enero de 2026. Es una versión mayor que cambia el comportamiento predeterminado de la biblioteca en varios puntos, por lo que código escrito para *Pandas* 2 puede dar resultados distintos o desencadenar excepciones.

| Aspecto | *Pandas* 2 | *Pandas* 3 |
| :--- | :--- | :--- |
| Texto | Columnas de tipo ```object``` de *NumPy*: pueden contener cualquier objeto de *Python*. | Tipo dedicado ```str```, respaldado por *PyArrow*: sólo admite cadenas y valores faltantes. |
| Copias y vistas | Un subconjunto podía ser vista o copia según el caso; de ahí el ```SettingWithCopyWarning```. | *Copy-on-Write* es el único modo: cualquier resultado de indexar se comporta como una copia. |
| Asignación encadenada | ```df['b'][filtro] = valor``` a veces modificaba el original. | Nunca modifica el original y emite un ```ChainedAssignmentError```. |
| ```serie[n]``` con índice de texto | Buscaba la posición ```n``` (con una advertencia). | Busca la etiqueta ```n```; la posición se indica con ```iloc[]```. |
| Fechas | Siempre en nanosegundos (```datetime64[ns]```). | La resolución se infiere; al leer texto suele ser microsegundos (```datetime64[us]```). |
| Expresiones de columna | Funciones ```lambda```. | Además, ```pd.col()```, con una sintaxis parecida a la de *Polars* y *PySpark*. |

Fuentes: [Novedades de *Pandas* 3.0](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html) y [guía de migración](https://pandas.pydata.org/docs/user_guide/migration.html).

* La siguiente celda muestra la versión de *Pandas* instalada.

In [ ]:
pd.__version__

### El tipo de dato ```str```.

* La siguiente celda crea una serie con cadenas de caracteres y un valor faltante. En *Pandas* 2 su tipo sería ```object```; en *Pandas* 3 es ```str``` y se almacena con *PyArrow*.

In [ ]:
textos = pd.Series(['norte', 'sur', None])
print(textos.dtype)
print(textos.dtype.storage)
textos

* Una columna ```str``` sólo admite cadenas o valores faltantes. La siguiente celda intentará asignar un entero y desencadenará una excepción de tipo ```TypeError```.

In [ ]:
textos[0] = 1

* Si se necesita el comportamiento anterior, la serie puede convertirse explícitamente a ```object```.

In [ ]:
textos.astype(object).dtype

### *Copy-on-Write*.

* La siguiente celda crea el *dataframe* ```ventas``` y un subconjunto ```altas``` con los renglones cuyas unidades son mayores a ```1```. Modificar ```altas``` no afecta a ```ventas```: el subconjunto se comporta como una copia.

In [ ]:
ventas = pd.DataFrame({'unidades': [1, 2, 3], 'precio': [10, 20, 30]})
altas = ventas[ventas['unidades'] > 1]
altas['precio'] = 0
ventas

* La asignación encadenada (dos pares de corchetes seguidos) tampoco modifica el original. *Pandas* 3 lo advierte con un ```ChainedAssignmentError```.

In [ ]:
ventas['precio'][ventas['unidades'] > 1] = 0
ventas

* La forma correcta es hacer la selección y la asignación en un solo paso con ```loc[]```.

In [ ]:
ventas.loc[ventas['unidades'] > 1, 'precio'] = 0
ventas

### Selección por posición y por etiqueta.

* Con un índice de texto, ```serie[1]``` busca la etiqueta ```1``` y no la posición, por lo que la siguiente celda desencadenará una excepción de tipo ```KeyError```.

In [ ]:
regiones = pd.Series([10, 20, 30], index=['norte', 'centro', 'sur'])
regiones[1]

* La posición se indica con ```iloc[]``` y la etiqueta con ```loc[]```.

In [ ]:
regiones.iloc[1], regiones.loc['centro']

### Resolución de fechas.

* La siguiente celda convierte texto a fechas. En *Pandas* 2 el resultado siempre era ```datetime64[ns]```; en *Pandas* 3 la resolución se infiere a partir de los datos.

In [ ]:
pd.to_datetime(['2026-01-15', '2026-02-15']).dtype

### Expresiones con ```pd.col()```.

* ```pd.col()``` hace referencia a una columna sin escribir una función ```lambda```. Las dos expresiones siguientes son equivalentes.

In [ ]:
ventas.assign(total=lambda df: df['unidades'] * df['precio'])

In [ ]:
ventas.assign(total=pd.col('unidades') * pd.col('precio'))

### Migrar código de *Pandas* 2.

La recomendación oficial es actualizar primero a *Pandas* 2.3, eliminar todas las advertencias y sólo entonces pasar a *Pandas* 3. En *Pandas* 2.3 se pueden activar por adelantado los dos cambios principales:

```python
pd.options.future.infer_string = True    # tipo str para el texto
pd.options.mode.copy_on_write = True     # Copy-on-Write
```

## Los *dataframes*.

Los *dataframes* representan el componente primordial tanto de *Pandas* como de *R* y de *Spark SQL*.

Los *dataframes* de *Pandas* se basan en los arreglos de *Numpy*, conformando arreglos de datos de 2 dimensiones compuesto por columnas y renglones.

### La clase ```pd.DataFrame```.

La clase ```pd.DataFrame``` se utiliza para crear los *dataframes* de *Pandas*. 

```
pd.DataFrame(data=<objeto>, index=<índices>, columns=<indices de columnas>)
```

Donde:

* ```<objeto>``` es un objeto con las siguientes características:
 * Un objeto de tipo ```dict```.
 * Un objeto de tipo ```tuple``` que contiene a otros objetos tipo ```tuple```.
 * Un arreglo de tipo ```numpy.ndarray``` de 2 dimensiones.
 * Otra instancia de ```pd.DataFrame```.
* ```<índices>``` es una colección de cadenas de caracteres que serán usadas como identificadores para cada renglón del *dataframe*.
* ```<columnas>``` es una colección de cadenas de caracteres que serán usadas como identificadores para cada columna del *dataframe*. 

La documentación de ```pd.DataFrame``` puede ser consultada en:

* https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html
* https://pandas.pydata.org/docs/reference/frame.html

**Nota:** Por convención se utilizará ```df``` para hacer referencia a un objeto instanciado de ```pd.DataFrame```.

**Ejemplo:**

* La siguiente celda regresará un *dataframe* a partir de una colección de objetos tipo ```tuple```.

In [ ]:
pd.DataFrame(data=[(0, 1, 2),
                   (1, 2, 3), 
                   (2, 3, 4), 
                   (3, 4, 5)])

* La siguiente creará al objeto de tipo ```dict``` al que se le asignará el nombre ```diccionarios```.

In [ ]:
diccionarios = {'py101':[10, 5, 33 ,45, 25, 22], 
                'py111':[0, 15, 21 , 30, 31, 11], 
                'py121':[15, 5, 1 ,10, 42, 21], 
                'py301':[20, 35, 3 ,15, 0, 0], }

* La siguiente celda regresará un *dataframe* a partir del objeto ```diccionarios```.
* En este caso, la clave de cada elemento de ```diccionario``` será el identificador de cada columna del *dataframe* y los elementos del objeto tipo ```list``` correspondientes a cada clave serán los elementos de dicha columna.

In [ ]:
pd.DataFrame(data=diccionarios)

* Se creará un *dataframe* a partir de un objeto creado con ```numpy.arange()```.

* La siguiente celda creará un arreglo de *Numpy* de forma ```(3, 3)``` con nombre ```matriz```.

In [ ]:
matriz = np.arange(9).reshape(3, 3)
matriz

* La siguiente celda regresará un *dataframe* a partir de el arreglo ```matriz```.

In [ ]:
pd.DataFrame(matriz)

### Índices en un *dataframe*.

Los *dataframes* de *Pandas* definen índices para cada renglón (eje ```0```) y a cada columna (eje ```1```) de un *dataframe*.

En caso de que no sean definidos, los índices serán números enteros positivos que inician en ```0``` que se incrementarán de uno en uno.

* *Pandas* identifica a los índices de los renglones simplemente como "índices".
* Tanto los índices (de los renglones) como los índices de las columnas son instancias de la clase ```pd.Index```.

https://pandas.pydata.org/docs/reference/api/pandas.Index.html

### Definición de identificadores de índices al crear un *dataframe*.

Al instanciar un objeto a partir de la clase ```pd.DataFrame``` es posible asignarle los identificadores de los índices por medio de los siguientes parámetros:

* ```index```, al que se le asignará un objeto iterable que contiene a su vez objetos ```str``` que corresponderán al identificador cada índice (de renglón) del *dataframe*. El objeto asignando a este parámetro será asignado al atributo ```df.index```.
* ```columns```, al que se le asignará un objeto iterable que contiene a su vez objetos ```str``` que corresponderán al identificador de cada índice de columna del *dataframe*. El objeto asignando a este parámetro será asignado al atributo ```df.columns```.

**Ejemplos:**

* La siguiente celda definirá un objeto de tipo ```tuple``` con nombre ```indice```.

In [ ]:
indice = ('enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio')

* La siguiente celda regresará un *dataframe* creado a partir del objeto ```diccionarios``` definido previamente y asignando ```índice``` al parámetro ```index```.

In [ ]:
pd.DataFrame(data=diccionarios, index=indice)

* La siguiente celda regresará un *dataframe* creado a partir de:
 * Los datos del objeto ```matriz```, definido previamente.
 * El objeto ```['uno','dos','tres']``` que será asignado como argumento para el parámetro ```index```.
 * El objeto ```['a', 'b', 'c']``` que será asignado como argumento para el parámetro ```columns```.

In [ ]:
pd.DataFrame(matriz,
             index=['uno','dos','tres'],
             columns=['a', 'b', 'c'])

### Selección de columnas de un *dataframe* mediante identificadores.

Los *dataframes* permiten obtener los datos de una columna usando el identificador de la columna de forma similar a una clave de un objeto de tipo ```dict```.

```
df[<col>]
```

Donde:

* ```<col>``` es un objeto de tipo ```str``` correspondiente al identificador de una columna del *dataframe*.

**Nota:** Cada columna de un *dataframe* es una serie de *Pandas*.

**Ejemplo:**

* Se creará el *dataframe* ``cursos`` a partir de un objeto de tipo ```dict```, por lo que las claves del objeto ```dict``` corresponderán los identificadores de las columnas del *dataframe*.

In [ ]:
cursos = pd.DataFrame({'py101':[10, 5, 33 ,45, 25, 22], 
         'py111':[0, 15, 21 , 30, 31, 11], 
         'py121':[15, 5, 1 ,10, 42, 21], 
         'py301':[20, 35, 3 ,15, 0, 0]},
         index=('enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio'))

In [ ]:
cursos

* La siguiente celda regresará la serie correspondiente a la columna con identificador ```'py121'```.

In [ ]:
cursos['py121']

In [ ]:
type(cursos['py121'])

### Selección de columnas de un *dataframe* mediante índices numéricos.

Si las columnas de un *dataframe* **NO tiene identificadores asignados,** es posible seleccionar una de ellas mediante su índice numérico.

```
df[<ncol>]
```
Donde:

* ```<ncol>``` es un objeto de tipo ```int``` correspondiente a un índice del *dataframe*.


**Nota:** En caso de que las columnas tengan identificadores asignados, se desencadenará un error de tipo ```KeyError``` si se usa in índice numérico.

**Ejemplos:**

* El *dataframe* ```datos``` contiene columnas con identificadores numéricos.

In [ ]:
datos = pd.DataFrame([[1, 2, 3, 4],
                      [5, 6, 7, 8], 
                      [9, 10, 11, 12]])

In [ ]:
datos

* La siguiente celda regresará la columna con índice ```2``` del *dataframe* ```datos```.

In [ ]:
datos[2]

* Se utilizará el *dataframe* ```cursos``` definido previamente. 

In [ ]:
cursos

* La siguiente celda intentará usar un índice numérico para identificar una columna, pero el *dataframe* ```cursos``` ya cuentan con identificadores de columna que son cadenas de caracteres, por lo que se desencadenará una excepción de tipo ```KeyError```.

In [ ]:
cursos[3]

### Selección de renglones de un *dataframe* mediante rangos de índices.

Para seleccionar los renglones de un *dataframe* mediante un rango de identificadores, se usa una sintaxis de rangos por medio de dos puntos ```:```. 

El resultado será un nuevo *dataframe*.

```
df[<id_inicio>:<id_fin>:<n>]
```

Donde:
* ```<id_inicio>``` es un objeto ```int``` o ```str``` correspondiente a un índice de renglón a partir del cual se iniciará el rango.
* ```<id_fin>``` es un objeto ```int``` o ```str``` correspondiente a un índice de renglón en el que finalizará el rango.
* ```<n>``` es el número de incrementos/decrementos en el rango.

**Ejemplos:** 

* Se utilizará el *dataframe* ```cursos``` definido previamente.

In [ ]:
cursos

* La siguiente celda regresará el renglón correspondiente al índice ```'enero'``` usando la sintaxis ```cursos[:1]```.

In [ ]:
cursos[:1]

* El objeto resultante es un *dataframe* de *Pandas*.

In [ ]:
type(cursos[:1])

* Dado que el objeto resultante de ejecutar ```cursos[:1]``` es un nuevo *dataframe*, es posible concatenar índices como ocurre en la siguiente celda, en la que se obtendrá una serie a partir de la columna ```'py101'``` de ```cursos[:1]```.

In [ ]:
cursos[:1]['py101']

* La siguiente celda regresará los renglones del objeto ```cursos``` correspondientes a los índices ```'enero'```, ```'marzo'``` y ```'mayo'``` usando la sintaxis ```cursos[::2]```.

In [ ]:
cursos[::2]

* La siguiente celda regresará una serie con todos los elementos de la columna ```'py111'``` del *dataframe* ```cursos[::2]```.

In [ ]:
cursos[::2]['py111']

* La siguiente celda regresará un nuevo *dataframe* obtenido a partir los renglones con los índices ```'marzo'```, ```'abril'``` y ```'mayo'``` del *dataframe* ```cursos```, usando la sintaxis ```cursos['marzo':'mayo']```.

In [ ]:
cursos['marzo':'mayo']

* La siguiente celda no corresponde a un rango, por lo que el identificador se aplicará a las columnas y desencadenará un error ```KeyError```.

In [ ]:
cursos['enero']

* El siguiente rango usa identificadores de columna, pero la notación ```df[inicio:fin]``` se aplica a los **renglones**. Como ```'py101'``` y ```'py121'``` no son identificadores de renglón, se desencadenará un error ```KeyError```.

In [ ]:
cursos['py101': 'py121']

### Resumen de la forma ```df[ ]```.

* Para acceder a las columnas es necesario especificar el identificador de la columna y si no existe el identificador, se puede usar el índice numérico correspondiente.

* Para acceder a los renglones, se deben de definir rangos y usarse los identificadores o los índices númericos de forma indistinta.

## Las series de *Pandas*.

Las series son objetos instanciado de la clase ```pandas.Series``` y son objetos de una sola dimensión.

```
pd.Series(<datos>, name=<nombre>, index=<índices>)
```

Donde:

* ```<datos>``` puede ser un objeto de tipo:
 * ```tuple```
 * ```list```
 * ```dict``` 
 * Un arreglo de *Numpy* de una dimensión.
* ```<nombre>``` es un objeto de tipo ```str``` y corresponderá al atributo ```name``` del objeto resultante.
* ```<índices>``` es una colección de cadenas de caracteres correspondientes al índice de cada elemento de la serie.


La documentación de las series de pandas puede ser consultada en:

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.html

**Ejemplos:**

* La siguiente celda creará una serie.

In [ ]:
pd.Series([12, 4, 32, 41, 33, 28], 
          name='py201', 
          index=('enero',
                 'febrero', 
                 'marzo', 
                 'abril', 
                 'mayo', 
                 'junio'))

* La siguiente celda creará una serie de nombre ```mi_serie```.

In [ ]:
mi_serie = pd.Series([12, 4, 32, 41, 33, 28], 
                  name='py201', 
                  index=('enero', 
                         'febrero', 
                         'marzo', 
                         'abril', 
                         'mayo', 
                         'junio'))

* La siguiente celda regresará el objeto ```str``` correspondiente al atributo ```name``` de la serie ```mi_serie```.

In [ ]:
mi_serie.name

* La siguiente celda regresará al elemento con índice ```'abril'``` de la serie ```mi_serie```.

In [ ]:
mi_serie['abril']

La siguiente celda regresará una serie creada a partir de ```mi_serie``` con los elementos dentro del rango ```[::2]```.

In [ ]:
mi_serie[::2]

La siguiente celda regresará una serie creada a partir de ```mi_serie``` con los elementos dentro del rango ```[enero:abril:2]```.

In [ ]:
mi_serie['enero':'mayo':2]

### Selección de renglones dentro de la columna de un *dataframe*.

Debido a que las columnas de un *dataframe* son series, es posible acceder a elementos específicos de *dataframe* utilizando la siguiente sintaxis:

```
df[<col>][<índice>]
```

Donde:

* ```<col>``` es el identificador de una columna existente en el *dataframe*.
* ```<índice>``` es el identificador de un índice o rango de índices existentes en el *dataframe*.

**Nota:** La sintaxis ```df[<col>][<índice>]``` puede ser usada para acceder a elementos específicos de un *dataframe* pero no es la forma recomendada, ya que puede desencadenar errores de tipo ```KeyError``` o ```IndexError``` si el identificador de la columna o el índice no existen en el *dataframe*. En su lugar, se recomienda usar los métodos ```df.loc[]``` o ```df.iloc[]``` para acceder a elementos específicos de un *dataframe*, los cuales se estudiarán más adelante.

**Ejemplo:**

* Se utilizará el *dataframe* ```cursos``` definido previamente.

In [ ]:
cursos

* La siguiente celda regresará una serie con el contenido de la columna con identificador ```py101```.

In [ ]:
cursos['py101']

* La siguiente celda regresará al elemento en la posición ```2``` de la serie ```cursos['py101']```. Como el índice de la serie está formado por cadenas de caracteres, la posición se indica con ```iloc[]```: a partir de *Pandas* 3, ```serie[2]``` busca la etiqueta ```2``` y no la posición.

In [ ]:
cursos['py101'].iloc[2]

* La siguiente celda regresará al elemento con índice ```'marzo'``` de la serie ```cursos['py101']```.

In [ ]:
cursos['py101']['marzo']

* Las siguientes celdas obtendrán del *dataframe* ```cursos``` el contenido de las celdas correspondiente a la columna con identificador ```py111``` y las celdas con los índices ```abril``` y ```mayo```.

In [ ]:
cursos['py111'][3:5]

* Las columnas no aceptan rangos y la siguiente celda desencadenará un error ```KeyError```.

In [ ]:
cursos['py101':'py111'][3:5]

### Selección de renglones de un *dataframe* a partir de una expresión lógica sobre una columna.

Es posible aprovechar el *broadcasting* para aplicar expresiones lógicas en una columna de un *dataframe* con la finalidad de seleccionar sólo los renglones en los que la expresión da por resultado ```True```.

```
df[<expresion>]
```

Donde:

* ```<expresion>``` es una expresión lógica aplicada sobre una columna de un *dataframe* que da por resultado una serie con valores booleanos. Esta serie tiene el nombre de una columna del *dataframe* y exactamente los mismos índices del dataframe. 

**Ejemplo:**

* Se utilizará el *dataframe* ```cursos```, definido previamente.

In [ ]:
cursos

* La siguiente celda aplicará broadcasting sobre la columna ```cursos[py121]``` y regresará una serie en la que validará si cada elemento de la columna es mayor que ```20```. Solamente los elementos con índices ```mayo``` y ```junio``` dan por resultado ```True```.

In [ ]:
cursos['py121'] > 20

* La siguiente celda aprovechará la expresión ```cursos['py121'] > 20``` para crear un *dataframe* que contiene exclusivamente los renglones en los que la expresión es ```True```.

In [ ]:
cursos[cursos['py121'] > 20]

### Conversión de series a *dataframes*. 

El método ```to_frame()``` de las series permite transformar una serie en un *dataframe* de una columna.

```
<serie>.to_frame()
```

**Ejemplo:**

In [ ]:
pd.Series([12, 4, 32, 41, 33, 28],
          index=indice, name='py201')

* La siguiente celda transformará una serie en un *dataframe*.

In [ ]:
pd.Series([12, 4, 32, 41, 33, 28],
          index=indice, name='py201').to_frame()

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>